In [2]:
import sys, os
dir = os.getcwd()

ext = ['', '/..', '/../src/models', '/../src/nlp', '/../src/synth']
sys.path += [dir + i for i in ext]

In [3]:
from constraint import *

In [ ]:
class ProgressivePosition(Constraint):
    """
    Purpose: Ensures that movement is progressive and helps penetrate defensive lines
    Coaching Link: "you're looking to receive balls in behind the line"
    
    This constraint ensures:
    1. Movement is forward-oriented (toward goal/objective)
    2. Position is beyond key defensive lines
    3. Creates advantageous attacking positions
    """
    def __init__(self, args):
        self.reference_line = args.get('reference_line', None)  # e.g., midfield line
        self.min_progress = args.get('min_progress', None)  # minimum forward distance
        
    def __call__(self, scene, sample):
        # Validates if position advances play beyond reference line (e.g., midfield)
        return sample.y > self.reference_line + self.min_progress

class SpaceSeparation(Constraint):
    """
    Purpose: Ensures adequate separation from opponents to receive safely
    Coaching Link: "don't receive the ball in front of these Midfield players"
    
    This constraint checks:
    1. Distance from all nearby opponents
    2. Safe space to receive ball
    3. Room to control and make next action
    """
    def __init__(self, args):
        self.min_distance = args.get('min_distance', None)  # Minimum safe distance from opponents
        self.opponent_type = args.get('opponent_type', 'opponent')
        
    def __call__(self, scene, sample):
        for obj in [obj for obj in scene if obj.type == self.opponent_type]:
            if distance(sample, obj.location) < self.min_distance:
                return False
        return True

class SupportAngle(Constraint):
    """
    Purpose: Ensures proper positioning to receive passes and continue play
    Coaching Link: "support forward well done"
    
    This constraint ensures:
    1. Player is at a good angle to receive pass
    2. Position allows for next action after receiving
    3. Maintains tactical shape with ball holder
    """
    def __init__(self, args):
        self.ball_holder = args.get('ball_holder', None)
        self.min_angle = args.get('min_angle', 45)  # Minimum angle for good support
        self.max_angle = args.get('max_angle', 135)  # Maximum angle for good support
        
    def __call__(self, scene, sample):
        angle = calculate_angle(self.ball_holder.location, sample)
        return self.min_angle <= angle <= self.max_angle

class ReceivingWindow(Constraint):
    """
    Purpose: Handles timing of runs and movement
    Coaching Link: "don't go too early" and "time your runs"
    
    This constraint ensures:
    1. Player arrives at right time to receive
    2. Movement is synchronized with play development
    3. Not too early or too late for the pass
    """
    def __init__(self, args):
        self.time_window = args.get('time_window', [0.5, 2.0])  # Valid arrival time window
        self.player_speed = args.get('player_speed', None)
        
    def __call__(self, scene, sample):
        time_to_position = distance(current_pos, sample) / self.player_speed
        return self.time_window[0] <= time_to_position <= self.time_window[1]

class TacticalConnectivity(Constraint):
    """
    Purpose: Maintains team shape and passing options
    Coaching Link: "let me hear here Set set set the ball up for him"
    
    This constraint ensures:
    1. Position maintains connection with teammates
    2. Creates passing options
    3. Supports team structure
    """
    def __init__(self, args):
        self.max_distance = args.get('max_distance', None)  # Maximum distance to maintain connection
        self.min_teammates = args.get('min_teammates', 2)  # Minimum connected teammates
        
    def __call__(self, scene, sample):
        connected_teammates = 0
        for teammate in [obj for obj in scene if obj.type == 'teammate']:
            if distance(sample, teammate.location) < self.max_distance:
                connected_teammates += 1
        return connected_teammates >= self.min_teammates

In [ ]:
from abc import ABC, abstractmethod
import numpy as np
from typing import Dict, List, Any, Tuple, Optional

class Constraint(ABC):
    @abstractmethod
    def __call__(self, scene: Dict[str, Any], sample: np.ndarray) -> bool:
        pass

class SpacingConstraint(Constraint):
    """
    Purpose: Maintains appropriate spacing between agents
    Coaching Link: "make sure you've got as much width as possible to stretch the other team"
    
    This constraint ensures:
    1. Minimum distance from other agents to prevent crowding
    2. Maximum distance from reference points to maintain system cohesion
    3. Optimal space utilization for maneuverability
    """
    def __init__(self, args: Dict[str, Any]):
        self.min_distance = args.get('min_distance', 5.0)  # Minimum distance from other agents
        self.max_distance = args.get('max_distance', 30.0)  # Maximum distance from reference point
        self.reference_point = args.get('reference_point', None)  # e.g., ball position, team centroid
        
    def __call__(self, scene: Dict[str, Any], sample: np.ndarray) -> bool:
        # Check minimum spacing from other agents
        for agent in scene['agents']:
            if np.linalg.norm(sample - agent.position) < self.min_distance:
                return False
        
        # Check maximum distance from reference point
        if self.reference_point is not None:
            if np.linalg.norm(sample - self.reference_point) > self.max_distance:
                return False
                
        return True

class LineOfSightConstraint(Constraint):
    """
    Purpose: Ensures clear vision and passing lanes
    Coaching Link: "make sure that kids are at least doing the basics and getting into the right positions"
    
    This constraint ensures:
    1. Clear line of sight to key reference points (ball, teammates)
    2. Unobstructed passing lanes
    3. Visual control of critical spaces
    """
    def __init__(self, args: Dict[str, Any]):
        self.key_points = args.get('key_points', [])  # List of points that must be visible
        self.obstruction_threshold = args.get('obstruction_threshold', 1.0)  # Minimum clear path width
        
    def __call__(self, scene: Dict[str, Any], sample: np.ndarray) -> bool:
        for point in self.key_points:
            # Vector from sample to key point
            direction = point - sample
            distance = np.linalg.norm(direction)
            normalized_direction = direction / distance
            
            # Check for obstructions along line of sight
            for obstacle in scene['obstacles']:
                if self._check_obstruction(sample, point, obstacle):
                    return False
                    
        return True
        
    def _check_obstruction(self, start: np.ndarray, end: np.ndarray, 
                          obstacle: Dict[str, Any]) -> bool:
        # Implementation of line-obstacle intersection check
        pass

class SupportNetworkConstraint(Constraint):
    """
    Purpose: Ensures formation of effective support structures
    Coaching Link: "those little routines will really help cement to kids exactly where to go"
    
    This constraint ensures:
    1. Formation of triangular support structures
    2. Multiple layers of support
    3. Balanced distribution of agents
    """
    def __init__(self, args: Dict[str, Any]):
        self.min_triangles = args.get('min_triangles', 2)  # Minimum number of support triangles
        self.max_edge_length = args.get('max_edge_length', 15.0)  # Maximum distance for support
        self.min_angle = args.get('min_angle', 30.0)  # Minimum angle in support triangle
        
    def __call__(self, scene: Dict[str, Any], sample: np.ndarray) -> bool:
        teammates = scene['teammates']
        triangles = 0
        
        # Count valid support triangles formed with new position
        for i in range(len(teammates)):
            for j in range(i + 1, len(teammates)):
                if self._forms_valid_triangle(sample, teammates[i].position, 
                                           teammates[j].position):
                    triangles += 1
                    
        return triangles >= self.min_triangles
        
    def _forms_valid_triangle(self, p1: np.ndarray, p2: np.ndarray, 
                            p3: np.ndarray) -> bool:
        # Check edge lengths
        edges = [
            np.linalg.norm(p2 - p1),
            np.linalg.norm(p3 - p2),
            np.linalg.norm(p1 - p3)
        ]
        if any(edge > self.max_edge_length for edge in edges):
            return False
            
        # Check angles
        angles = []
        for i in range(3):
            v1 = np.roll(edges, i)[0]
            v2 = np.roll(edges, i)[1]
            angle = np.arccos((v1**2 + v2**2 - np.roll(edges, i)[2]**2) / (2*v1*v2))
            angles.append(np.degrees(angle))
            
        return all(angle >= self.min_angle for angle in angles)

class DynamicResponseConstraint(Constraint):
    """
    Purpose: Ensures position allows for adaptive responses
    Coaching Link: "don't tell them they've made the mistake tell them that you like what they were doing"
    
    This constraint ensures:
    1. Ability to respond to state changes
    2. Maintenance of recovery options
    3. Energy-efficient positioning
    """
    def __init__(self, args: Dict[str, Any]):
        self.safe_zone = args.get('safe_zone', None)  # Recovery position reference
        self.max_response_time = args.get('max_response_time', 2.0)  # Maximum time to respond
        self.velocity = args.get('velocity', 5.0)  # Agent movement speed
        
    def __call__(self, scene: Dict[str, Any], sample: np.ndarray) -> bool:
        # Check if recovery position is reachable within time limit
        if self.safe_zone is not None:
            time_to_recover = np.linalg.norm(sample - self.safe_zone) / self.velocity
            if time_to_recover > self.max_response_time:
                return False
                
        # Check if position allows responses to likely state changes
        for threat in scene.get('threats', []):
            if not self._can_respond_to_threat(sample, threat):
                return False
                
        return True
        
    def _can_respond_to_threat(self, position: np.ndarray, 
                             threat: Dict[str, Any]) -> bool:
        # Implementation of threat response check
        pass

class ProgressionConstraint(Constraint):
    """
    Purpose: Ensures position enables system progression
    Coaching Link: "if they manage to win the ball then some learning goes on"
    
    This constraint ensures:
    1. Position advances system state
    2. Maintains balance between progression and security
    3. Creates advantageous future states
    """
    def __init__(self, args: Dict[str, Any]):
        self.objective_point = args.get('objective_point', None)  # Target/goal position
        self.min_progress = args.get('min_progress', 0.0)  # Minimum forward progress required
        self.max_risk = args.get('max_risk', 0.7)  # Maximum acceptable risk level
        
    def __call__(self, scene: Dict[str, Any], sample: np.ndarray) -> bool:
        if self.objective_point is not None:
            # Calculate progress toward objective
            current_distance = np.linalg.norm(scene['current_position'] - self.objective_point)
            new_distance = np.linalg.norm(sample - self.objective_point)
            progress = current_distance - new_distance
            
            if progress < self.min_progress:
                return False
            
            # Assess risk level of position
            risk = self._calculate_risk(scene, sample)
            if risk > self.max_risk:
                return False
                
        return True
        
    def _calculate_risk(self, scene: Dict[str, Any], 
                       position: np.ndarray) -> float:
        # Implementation of risk calculation
        pass

In [ ]:
class OptimalDistance(Constraint):
    """
    Purpose: Maintains effective operational distance from target/opponent
    Coaching Link: "be at arm's length from him" and "don't wrestle them"
    
    This constraint ensures:
    1. Not too close to lose maneuverability
    2. Not too far to lose effectiveness
    3. Maintains tactical advantage
    """
    def __init__(self, args):
        self.min_distance = args.get('min_distance', 1.0)  # Minimum effective distance
        self.max_distance = args.get('max_distance', 3.0)  # Maximum effective distance
        self.target_type = args.get('target_type', 'opponent')
        
    def __call__(self, scene, sample):
        for obj in [obj for obj in scene if obj.type == self.target_type]:
            dist = distance(sample, obj.location)
            if not (self.min_distance <= dist <= self.max_distance):
                return False
        return True

class BodyOrientation(Constraint):
    """
    Purpose: Ensures proper body positioning for maximum awareness and options
    Coaching Link: "don't be like this [square]" and "be on the half turn"
    
    This constraint ensures:
    1. Maintains visibility of both ball and space
    2. Body position enables quick reactions
    3. Maximizes available options
    """
    def __init__(self, args):
        self.min_angle = args.get('min_angle', 30)  # Minimum angle from square position
        self.max_angle = args.get('max_angle', 60)  # Maximum angle from square position
        self.reference_point = args.get('reference_point', None)  # Ball or opponent position
        
    def __call__(self, scene, sample):
        orientation_angle = calculate_orientation_angle(
            sample.orientation,
            self.reference_point
        )
        return self.min_angle <= orientation_angle <= self.max_angle

class MomentumPreservation(Constraint):
    """
    Purpose: Maintains appropriate movement dynamics
    Coaching Link: "don't slow it down" and "run at full pace"
    
    This constraint ensures:
    1. Maintains advantageous momentum
    2. Speed appropriate for situation
    3. Enables effective execution of next action
    """
    def __init__(self, args):
        self.min_velocity = args.get('min_velocity', 0.5)  # Minimum effective velocity
        self.max_velocity = args.get('max_velocity', 1.0)  # Maximum effective velocity
        self.direction_tolerance = args.get('direction_tolerance', 30)  # Degrees
        
    def __call__(self, scene, sample):
        current_velocity = calculate_velocity(scene.agent)
        velocity_magnitude = magnitude(current_velocity)
        velocity_direction = direction(current_velocity, sample)
        
        return (self.min_velocity <= velocity_magnitude <= self.max_velocity and
                abs(velocity_direction) <= self.direction_tolerance)

class ActionSpace(Constraint):
    """
    Purpose: Ensures position enables future actions while limiting opponent options
    Coaching Link: "safe side" and "make it hard for him"
    
    This constraint ensures:
    1. Position enables multiple future options
    2. Restricts opponent's effective choices
    3. Maintains tactical advantage
    """
    def __init__(self, args):
        self.min_options = args.get('min_options', 2)  # Minimum available actions
        self.safe_side_angle = args.get('safe_side_angle', 45)  # Angle for safe side positioning
        self.space_factor = args.get('space_factor', 1.5)  # Multiplier for required space
        
    def __call__(self, scene, sample):
        available_actions = calculate_available_actions(scene, sample)
        opponent_actions = calculate_opponent_actions(scene, sample)
        safe_side = is_safe_side(sample, scene.ball, scene.opponent, self.safe_side_angle)
        
        return (len(available_actions) >= self.min_options and
                len(opponent_actions) < len(available_actions) and
                safe_side)

**Distance**
<br/>
(distance) from (object)
<br/>
learn range

**Orientation**
<br/>
(Orientation) from (object)
<br/>
learnt range

**AheadOfLine**
<br/>
True ahead of (line)
<br/>
learnt height of line across axis (axis by LLM)

**CanShoot**
<br/>
True if (player) can shoot

**CanMakePass**
<br/>
True if (player) can pass

**Does compete formation**
<br/>
Learn shape